# yt-music-agent on Colab (ACE-Step, no Suno)

Runs the whole daily batch on Colab's free GPU: music with **ACE-Step**
(Apache-2.0, open weights), artwork, cinematic thumbnails, 1080p videos,
titles/descriptions/tags, and a zip you download at the end.

**Before running:** Runtime -> Change runtime type -> **T4 GPU**.

Notes
- First run downloads ~8GB of model weights (a few minutes).
- ACE-Step is strongest on instrumental / ambient / raga style tracks. It does
  not sing Hindi lyrics anywhere near Suno quality - keep vocal songs on Suno
  (`--music suno`, from your own machine) and use this for the long
  instrumental uploads.
- Nothing is uploaded to YouTube; you publish manually.

## 1. Check the GPU

In [ ]:
!nvidia-smi || echo 'NO GPU: Runtime -> Change runtime type -> T4 GPU'

## 2. Install the agent, ffmpeg and ACE-Step (~5 min)

In [ ]:
%%bash
set -e
apt-get -qq install -y ffmpeg > /dev/null
BRANCH=${BRANCH:-main}
[ -d /content/Yt-music-agent ] || git clone -q https://github.com/coolcryptomaniac/Yt-music-agent.git /content/Yt-music-agent
git -C /content/Yt-music-agent fetch -q origin "$BRANCH" && git -C /content/Yt-music-agent checkout -q FETCH_HEAD
pip install -q -r /content/Yt-music-agent/requirements.txt
pip install -q git+https://github.com/ace-step/ACE-Step.git
echo installed

## 3. Keys (optional)

A free Gemini key (https://aistudio.google.com/apikey) writes much better
titles/descriptions. Without any key the agent falls back to offline
templates and keyless Pollinations artwork, so it still runs.

In [ ]:
import os

os.environ["GEMINI_API_KEY"] = ""  # optional
os.environ["CEREBRAS_API_KEY"] = ""  # optional
os.environ["GROQ_API_KEY"] = ""  # optional

## 4. Batch settings

`COUNT` tracks, each `DURATION` seconds. On a T4 expect roughly 2-4 minutes
of GPU time per 2.5-minute track, plus ~1 minute of rendering.

In [ ]:
COUNT = 6
DURATION = 150
NICHE = (
    "Long meditative Indian classical and ambient instrumentals - raga flute, "
    "sitar, tanpura drone, monsoon rain, temple bells. Calm, cinematic, no vocals."
)
ARTIST = "Mohit Pandey"

## 5. Run the batch

In [ ]:
import os
import sys

os.chdir("/content/Yt-music-agent")
sys.path.insert(0, "/content/Yt-music-agent")

import logging

from ytmusic.config import load_config
from ytmusic.pipeline import produce_batch

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

config = load_config("config.yaml")
config.set("music.provider", "acestep")
config.set("music.instrumental", True)
config.set("acestep.duration", DURATION)
config.set("music.min_duration", min(90, DURATION))
config.set("content.mix", ["instrumental"])
config.set("channel.niche", NICHE)
config.set("channel.artist", ARTIST)

results = produce_batch(config, count=COUNT)
for item in results:
    print(item.plan.index, item.plan.youtube_title, item.video, item.notes)

## 6. Preview one result

In [ ]:
from IPython.display import Audio, Image, display

first = results[0]
if first.thumbnail:
    display(Image(filename=str(first.thumbnail)))
if first.audio:
    display(Audio(filename=str(first.audio)))
print(first.plan.youtube_title)
print(first.plan.description)

## 7. Zip and download (optionally save to Drive)

In [ ]:
import shutil
from pathlib import Path

SAVE_TO_DRIVE = False

batch_dir = results[0].directory.parent
archive = shutil.make_archive(f"/content/{batch_dir.name}", "zip", batch_dir)
print(archive, round(Path(archive).stat().st_size / 1e6, 1), "MB")

if SAVE_TO_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    target = Path("/content/drive/MyDrive/yt-music-agent")
    target.mkdir(parents=True, exist_ok=True)
    shutil.copy(archive, target)
    print("copied to", target)
else:
    from google.colab import files

    files.download(archive)

Each track folder contains `video.mp4`, `thumbnail.jpg`, `cover.jpg`, the audio
and `metadata.txt` (title, description, tags, chapters). `UPLOAD.md` in the
batch root is the paste-into-YouTube checklist.